# 02 特徵工程（成員 A）

特徵步驟寫在 `src/features.py`。這本 notebook 用來：
1. 檢查每個步驟的結果是否正確
2. 試新特徵
3. 做 ablation，回答「哪個特徵工程最有用？」

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))  # 讓 notebook 能 import src/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from src.config import FIGURE_DIR, TARGET
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
sns.set_theme(style="whitegrid")

In [ ]:
from src.data import load_data
from src.features import STEPS, build_features
train, test = load_data()
print("可用步驟：", list(STEPS))

## 1. 逐步檢查

In [ ]:
steps = []
for name in STEPS:
    steps.append(name)
    X_tr, X_te = build_features(train, test, steps)
    print(f"{' + '.join(steps):<70} → 特徵數 {X_tr.shape[1]}, NaN={X_tr.isna().sum().sum()}")

In [ ]:
# 確認新特徵和房價的關係
X_tr, _ = build_features(train, test, ["fix_types", "fill_missing", "encode_ordinal", "add_features"])
new = ["TotalSF", "TotalBath", "HouseAge", "RemodAge", "TotalPorchSF"]
X_tr[new].corrwith(np.log1p(train[TARGET])).sort_values(ascending=False)

In [ ]:
plt.figure(figsize=(6, 4))
sns.scatterplot(x=X_tr["TotalSF"], y=train[TARGET])
plt.title("TotalSF vs SalePrice"); plt.tight_layout(); plt.savefig(FIGURE_DIR / "totalsf_vs_price.png", dpi=150)

## 2. 試新特徵

先在這裡試，有效的話搬進 `src/features.py`（寫成新函式並加進 `STEPS`），再請 B 在 `versions.py` 的 iter3 加上它。

In [ ]:
# 例：
# def add_my_feature(df):
#     df = df.copy()
#     df["QualxArea"] = df["OverallQual"] * df["GrLivArea"]
#     return df

## 3. Ablation：哪個步驟最有用？

`delta` 越大 = 拿掉後分數變越差 = 該步驟越重要。結果也存到 `reports/ablation_log.csv`。

In [ ]:
from src.pipeline import ablation
result = ablation("iter3_features")
result

In [ ]:
r = result[result.removed_step != "(none)"]
plt.figure(figsize=(7, 3)); sns.barplot(data=r, x="delta", y="removed_step")
plt.title("CV RMSE increase when step is removed"); plt.tight_layout()
plt.savefig(FIGURE_DIR / "ablation.png", dpi=150)

## 結論（寫進報告：最有用的特徵工程 + 原因）

- 